# PrimeKG Graph Exploration

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import config
import pandas as pd
import networkx as nx

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Load data

In [ ]:
if not config.PRIMEKG_CSV.exists():
    print(f"File not found: {config.PRIMEKG_CSV}")
else:
    df_kg = pd.read_csv(config.PRIMEKG_CSV, low_memory=False)
    print(f"Rows: {len(df_kg):,}, Columns: {len(df_kg.columns)}")
    display(df_kg.head())

## Summary and missing values

In [ ]:
if 'df_kg' in locals():
    summary = pd.DataFrame({
        'dtype': df_kg.dtypes,
        'null_count': df_kg.isnull().sum(),
        'null_pct': (df_kg.isnull().sum() / len(df_kg) * 100).round(2)
    })
    display(summary)

## Node types

In [ ]:
if 'df_kg' in locals():
    print("Source node types (x_type):")
    print(df_kg['x_type'].value_counts())
    
    print("\nTarget node types (y_type):")
    print(df_kg['y_type'].value_counts())
    
    # Edges where both source and target are disease or phenotype
    mask = (
        df_kg['x_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES) &
        df_kg['y_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES)
    )
    print(f"\nDisease/phenotype edges: {mask.sum():,} ({mask.mean() * 100:.1f}%)")

## Relation distribution

In [ ]:
if 'df_kg' in locals():
    rel_counts = df_kg['relation'].value_counts()
    display(rel_counts.to_frame(name="count"))
    
    print("\nRelation whitelist check:")
    for rel in config.PRIMEKG_RELATION_WHITELIST:
        count = rel_counts.get(rel, 0)
        print(f"  {rel:<20}: {count:,} edges")

## Keyword search for clinical entities

In [ ]:
if 'df_kg' in locals():
    terms = ["sepsis", "pneumonia", "hypotension", "shock", "lactate"]
    
    for term in terms:
        matches = df_kg[
            (df_kg['x_type'].isin(['disease', 'effect/phenotype'])) &
            (df_kg['x_name'].str.contains(term, case=False, na=False))
        ]['x_name'].unique()
        
        print(f"{term} ({len(matches)} matches):")
        for m in matches[:5]:
            print(f"  {m}")
        print()

## Graph construction and path check

In [ ]:
if 'df_kg' in locals():
    # Filter edges by whitelist and node types
    clinical_df = df_kg[
        df_kg['relation'].isin(config.PRIMEKG_RELATION_WHITELIST) &
        df_kg['x_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES) &
        df_kg['y_type'].isin(config.PRIMEKG_ALLOWED_NODE_TYPES)
    ]
    
    # Build directed graph from dataframe
    G = nx.from_pandas_edgelist(
        clinical_df,
        source="x_name",
        target="y_name",
        edge_attr=["relation"],
        create_using=nx.DiGraph()
    )
    print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
    
    # Check path between sample entities
    node_map = {str(n).lower(): n for n in G.nodes()}
    src = next((n for k, n in node_map.items() if "bacterial pneumonia" in k), None)
    tgt = next((n for k, n in node_map.items() if "septic shock" in k), None)
    
    if src and tgt:
        G_undir = G.to_undirected()
        if nx.has_path(G_undir, src, tgt):
            path = nx.shortest_path(G_undir, src, tgt)
            print(f"\nShortest path ({len(path) - 1} hops):")
            for i in range(len(path) - 1):
                u, v = path[i], path[i + 1]
                edge = G.get_edge_data(u, v) or G.get_edge_data(v, u) or {}
                rel = edge.get('relation', 'connected_to')
                print(f"  {u} --[{rel}]--> {v}")
        else:
            print(f"No path found between '{src}' and '{tgt}'")